In [1]:
## Use google/gemma-4-12B-it
## to create results for image based problems

from google.colab import userdata
from huggingface_hub import login
import json
import time
import requests
import torch
from PIL import Image
from transformers import AutoProcessor, AutoModelForMultimodalLM

In [2]:
## check if cuda is available and GPU

print("CUDA available:", torch.cuda.is_available())
print("Using GPU:", torch.cuda.get_device_name(0))

CUDA available: True
Using GPU: NVIDIA A100-SXM4-40GB


In [3]:
## authenticate with huggingface

hf_token = userdata.get("hf_token")
login(token=hf_token)

In [4]:
model_name ="gemma-4"
model_id = "google/gemma-4-12B-it"

# Load model
processor = AutoProcessor.from_pretrained(model_id)
model = AutoModelForMultimodalLM.from_pretrained(
    model_id,
    dtype="auto",
    device_map="auto"
)

processor_config.json:   0%|          | 0.00/1.38k [00:00<?, ?B/s]

chat_template.jinja:   0%|          | 0.00/18.7k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/4.42k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/3.09k [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 32.2MB            

tokenizer.json: downloading bytes:           |  0.00B            

model.safetensors: reconstructing file:   0%|          |  0.00B / 23.9GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/677 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/260 [00:00<?, ?B/s]

In [5]:

## import the prompts from the image_prompts.py

import sys

sys.path.append("/content/drive/MyDrive/colab_files/Problems/Image")
from image_prompts import regular_system_prompt
from image_prompts import detail_system_prompt

print(regular_system_prompt)


{'Problem_1_Regular_Q': 'You are a physics expert. From the figure provide a brief description and calculate the total time taken by the block to travel 5 meters. All the necessary values required are given in the picture.', 'Problem_1_Obvious_Q': 'You are a physics expert. From the figure provide a brief description and calculate the total time taken by the block to travel 5 meters. All the necessary values required are given in the picture.', 'Problem_1_non_obvious_Q': 'You are a physics expert. From the figure provide a brief description and calculate the total time taken by the block to travel 5 meters. All the necessary values required are given in the picture.', 'Problem_2_Regular_Q': 'You are a physics expert. From the figure provide a brief description and calculate the time period of oscillation for small displacements. All the necessary values required are given in the picture.', 'Problem_2_Obvious_Q': 'You are a physics expert. From the figure provide a brief description and

In [6]:
print(detail_system_prompt)


{'Problem_1_Obvious_Q': 'You are a physics expert. From the figure provide a brief description and calculate the total time taken by the block to travel 5 meters. All the necessary values required are given in the picture. But if there are any physical laws broken then mention it and also say of the problem can be solved or not.', 'Problem_1_non_obvious_Q': 'You are a physics expert. From the figure provide a brief description and calculate the total time taken by the block to travel 5 meters. All the necessary values required are given in the picture. But if there are any physical laws broken then mention it and also say of the problem can be solved or not.', 'Problem_2_Obvious_Q': 'You are a physics expert. From the figure provide a brief description and calculate the time period of oscillation for small displacements. All the necessary values required are given in the picture. But if there are any physical laws broken then mention it and also say of the problem can be solved or not.

In [7]:
print(regular_system_prompt.keys())
print(detail_system_prompt.keys())

dict_keys(['Problem_1_Regular_Q', 'Problem_1_Obvious_Q', 'Problem_1_non_obvious_Q', 'Problem_2_Regular_Q', 'Problem_2_Obvious_Q', 'Problem_2_non_obvious_Q', 'Problem_3_Regular_Q', 'Problem_3_Obvious_Q', 'Problem_3_non_obvious_Q', 'Problem_4_Regular_Q', 'Problem_4_Obvious_Q', 'Problem_4_non_obvious_Q', 'Problem_5_Regular_Q', 'Problem_5_Obvious_Q', 'Problem_5_non_obvious_Q', 'Problem_6_Regular_Q', 'Problem_6_Obvious_Q', 'Problem_6_non_obvious_Q', 'Problem_7_Regular_Q', 'Problem_7_Obvious_Q', 'Problem_7_non_obvious_Q', 'Problem_8_Regular_Q', 'Problem_8_Obvious_Q', 'Problem_8_non_obvious_Q'])
dict_keys(['Problem_1_Obvious_Q', 'Problem_1_non_obvious_Q', 'Problem_2_Obvious_Q', 'Problem_2_non_obvious_Q', 'Problem_3_Obvious_Q', 'Problem_3_non_obvious_Q', 'Problem_4_Obvious_Q', 'Problem_4_non_obvious_Q', 'Problem_5_Obvious_Q', 'Problem_5_non_obvious_Q', 'Problem_6_Obvious_Q', 'Problem_6_non_obvious_Q', 'Problem_7_Obvious_Q', 'Problem_7_non_obvious_Q', 'Problem_8_Obvious_Q', 'Problem_8_non_obvio

In [8]:
from pathlib import Path

image_folder = Path("/content/drive/MyDrive/colab_files/Problems/Image")
file_names = sorted(file.name for file in image_folder.glob("*.png"))
print("Total number of files:::",len(file_names))
print(file_names)

Total number of files::: 24
['Problem_1_Obvious_Q.png', 'Problem_1_Regular_Q.png', 'Problem_1_non_obvious_Q.png', 'Problem_2_Obvious_Q.png', 'Problem_2_Regular_Q.png', 'Problem_2_non_obvious_Q.png', 'Problem_3_Obvious_Q.png', 'Problem_3_Regular_Q.png', 'Problem_3_non_obvious_Q.png', 'Problem_4_Obvious_Q.png', 'Problem_4_Regular_Q.png', 'Problem_4_non_obvious_Q.png', 'Problem_5_Obvious_Q.png', 'Problem_5_Regular_Q.png', 'Problem_5_non_obvious_Q.png', 'Problem_6_Obvious_Q.png', 'Problem_6_Regular_Q.png', 'Problem_6_non_obvious_Q.png', 'Problem_7_Obvious_Q.png', 'Problem_7_Regular_Q.png', 'Problem_7_non_obvious_Q.png', 'Problem_8_Obvious_Q.png', 'Problem_8_Regular_Q.png', 'Problem_8_non_obvious_Q.png']


In [9]:
## collect the dictionary to loop through

prompt_dict = [regular_system_prompt, detail_system_prompt]

In [10]:
## to calculate the total time taken to generate the output

def synchronize_cuda():
    # GPU operations are asynchronous.
    # Wait for pending operations before taking timing boundaries.
    if torch.cuda.is_available():
        for device_index in range(torch.cuda.device_count()):
            torch.cuda.synchronize(device_index)


In [12]:
%%time

for dicts in prompt_dict:
  for file in file_names:
    image_folder = Path("/content/drive/MyDrive/colab_files/Problems/Image")
    file_path = image_folder/ file

    if dicts is detail_system_prompt and "regular" in file_path.stem.lower():  ## skip detail_system_prompt if question has
      continue                                   ## the word regular in it, as we dont need detail system prompt for regular problem

    file_key = file_path.stem   ## get the name of the file, only the name

    if file_key not in dicts:   ## test if the file name is in the dictionary key or not. Note that the dictionary key and
      print(f"For {file_key} no dictionary key was found. Some error in file naming.")     ## file name is matched
      continue

    system_prompt = dicts[file_key]
    image = Image.open(file_path).convert("RGB")


    messages = [
        {
          "role": "user",
          "content": [
              {"type": "image", "image": image},
              {"type": "text", "text": system_prompt},
          ],
          }
        ]


    inputs = processor.apply_chat_template(
    messages,
    tokenize=True,
    add_generation_prompt=True,
    enable_thinking=False,
    return_dict=True,
    return_tensors="pt",
            ).to(model.device)

    input_length = inputs["input_ids"].shape[-1]



    ## generate and measure time
    synchronize_cuda()
    start_time = time.perf_counter()

    # Generate the response
    with torch.inference_mode():
      outputs = model.generate(
          **inputs,
          max_new_tokens=10000,
          do_sample=False,
            )

    synchronize_cuda()
    end_time = time.perf_counter()

    ## calculate time taken
    total_time_seconds = end_time - start_time

    generated_tokens = outputs[0, input_length:] ## remove input tokens to keep only newly generated tokens
    output_tokens = generated_tokens.numel() ## count the output tokens

    response = processor.decode(
    generated_tokens,
    skip_special_tokens=True,
        )
    output_text = response.strip()

    if dicts is detail_system_prompt:
      end_name="detail_system_prompt"
    else:
      end_name= "regular_system_prompt"

    output_filename= f"{model_name}_{file_path.stem}_{end_name}" +".md"

    output_folder = Path("/content/drive/MyDrive/colab_files/Results/gemma-4")
    output_file_path = output_folder / output_filename
    output_file_path.write_text(output_text, encoding="utf-8")


    ## Build metadata

    metadata = {
    "requested_model": model_id,
    "problem_question": file_key,
    "prompt_name": system_prompt,
    "total_output_tokens": output_tokens,
    "total_time_taken": total_time_seconds
    }

    metadata_folder= Path("/content/drive/MyDrive/colab_files/Results/gemma-4/metadata")
    metadata_name= f"{model_name}_{file_path.stem}_{end_name}" +".json"

    meta_data_file_path = metadata_folder / metadata_name
    meta_data_file_path.write_text(
                        json.dumps(metadata, indent=4, ensure_ascii=False),
                        encoding="utf-8",
                    )

    print(f"Completed:::{model_name}_{file_path.stem}_{end_name}")

Completed:::gemma-4_Problem_1_Obvious_Q_regular_system_prompt
Completed:::gemma-4_Problem_1_Regular_Q_regular_system_prompt
Completed:::gemma-4_Problem_1_non_obvious_Q_regular_system_prompt
Completed:::gemma-4_Problem_2_Obvious_Q_regular_system_prompt
Completed:::gemma-4_Problem_2_Regular_Q_regular_system_prompt
Completed:::gemma-4_Problem_2_non_obvious_Q_regular_system_prompt
Completed:::gemma-4_Problem_3_Obvious_Q_regular_system_prompt
Completed:::gemma-4_Problem_3_Regular_Q_regular_system_prompt
Completed:::gemma-4_Problem_3_non_obvious_Q_regular_system_prompt
Completed:::gemma-4_Problem_4_Obvious_Q_regular_system_prompt
Completed:::gemma-4_Problem_4_Regular_Q_regular_system_prompt
Completed:::gemma-4_Problem_4_non_obvious_Q_regular_system_prompt
Completed:::gemma-4_Problem_5_Obvious_Q_regular_system_prompt
Completed:::gemma-4_Problem_5_Regular_Q_regular_system_prompt
Completed:::gemma-4_Problem_5_non_obvious_Q_regular_system_prompt
Completed:::gemma-4_Problem_6_Obvious_Q_regular_sy